# Entity ***Liquidity Providers***
+ Join between **mints** and **burns**
+ Layer **gold**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

In [0]:
table_name = f"fact_{entity_name}"

### Execution

In [0]:
gld_lqe_df = spark.sql(f"""
    WITH cte_union_all AS (
        SELECT
            pk_mint_id AS pk_event_id,
            pool_id,
            token0_id,
            token1_id,
            tx_id,
            tick_lower_id,
            tick_upper_id,
            owner_address,
            sender_address,
            origin_address,
            token0_amount,
            token1_amount,
            usd_amount,
            timestamp,
            "mint" AS event_type
        FROM uniswap.silver.mints
        UNION ALL
        SELECT
            pk_burn_id AS pk_event_id,
            pool_id,
            token0_id,
            token1_id,
            tx_id,
            tick_lower_id,
            tick_upper_id,
            owner_address,
            NULL AS sender_address,
            origin_address,
            token0_amount,
            token1_amount,
            usd_amount,
            timestamp,
            "burn" AS event_type
        FROM uniswap.silver.burns
    )
    SELECT
        *,
        CASE
            WHEN event_type = "mint" THEN usd_amount
            WHEN event_type = "burn" THEN -usd_amount
        END AS liquidity_delta_usd,
        CASE
            WHEN usd_amount <= 1000 THEN "small"
            WHEN usd_amount <= 10000 THEN "medium"
            WHEN usd_amount <= 100000 THEN "large"
            ELSE "whale"
        END AS event_size_category,
        CURRENT_TIMESTAMP() AS _load_timestamp_gold
    FROM cte_union_all
""")

### Save & exit

In [0]:
load_result = load_entity(gld_lqe_df,
                        table_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)